In [ ]:
# Q.1) Basic TensorFlow 2 tensor operations

import tensorflow as tf

a, b = tf.constant([1, 2, 3]), tf.constant([4, 5, 6])
print("Addition:", a + b)
print("Reshaped:", tf.reshape(a + b, [3, 1]))
print("Result:", tf.constant(5.0) * tf.constant(2.0) + 3)
print("Eager execution:", tf.executing_eagerly())

@tf.function
def graph_fn(x, y):
    return x + y

print("Graph mode result:", graph_fn(a, b))


In [ ]:
# Q.2) Preprocess a small dataset (no external file)

import pandas as pd, numpy as np

df = pd.DataFrame({"Age": [20, 22, np.nan, 25], "Score": [80, 90, 70, np.nan], "Gender": ["M", "F", "M", "F"]})
print("Original:\n", df)

df = df.fillna(df.mean(numeric_only=True))
df[["Age", "Score"]] = df[["Age", "Score"]] / df[["Age", "Score"]].max()
df["Gender"] = df["Gender"].map({"M": 0, "F": 1})

print("\nCleaned:\n", df)


In [ ]:
# Q.2) Preprocess a small real dataset

import pandas as pd

df = pd.read_csv("Admission_Predict.csv")
print("Original:\n", df.head())

df = df.fillna(df.mean(numeric_only=True))
cols = ["GRE Score", "TOEFL Score", "CGPA"]
df[cols] = df[cols] / df[cols].max()
df["Research"] = df["Research"].astype(int)  # already 0/1 encoded

print("\nCleaned:\n", df.head())


In [ ]:
# Q.3) Visualize a small dataset

import pandas as pd, matplotlib.pyplot as plt, seaborn as sns

df = pd.DataFrame({"Age": [20, 22, 21, 25, 24, 23], "Score": [70, 80, 75, 90, 85, 88]})

df["Score"].plot(kind="hist", title="Score Distribution"); plt.show()
plt.scatter(df["Age"], df["Score"]); plt.xlabel("Age"); plt.ylabel("Score"); plt.title("Age vs Score"); plt.show()
sns.heatmap(df.corr(), annot=True); plt.title("Correlation Heatmap"); plt.show()


In [ ]:
# Q.3) Visualize a real dataset

import pandas as pd, matplotlib.pyplot as plt, seaborn as sns

df = pd.read_csv("Admission_Predict.csv")
print(df.head())

df["CGPA"].plot(kind="hist", title="CGPA Distribution"); plt.show()
plt.scatter(df["GRE Score"], df["CGPA"]); plt.xlabel("GRE Score"); plt.ylabel("CGPA"); plt.title("GRE vs CGPA"); plt.show()
sns.heatmap(df.corr(), annot=True); plt.title("Correlation Heatmap"); plt.show()


In [ ]:
# Q.4) Word embeddings + FAISS similarity search
# Note: !pip install gensim faiss-cpu

from gensim.models import Word2Vec
import faiss, numpy as np

sentences = [["cat", "dog", "pet"], ["cat", "animal"], ["dog", "animal"]]
model = Word2Vec(sentences, vector_size=5, min_count=1)

words = ["cat", "dog", "pet", "animal"]
vecs = np.array([model.wv[w] for w in words], dtype="float32")

index = faiss.IndexFlatL2(5)
index.add(vecs)

dist, idx = index.search(np.array([model.wv["cat"]], dtype="float32"), 2)
print("Similar to 'cat':", [words[i] for i in idx[0]])
print("Cat-Dog similarity:", model.wv.similarity("cat", "dog"))


In [ ]:
# Q.5) Diffusion model inference on MNIST (TensorFlow 2, run on Google Colab)

import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
from tensorflow.keras import layers

(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = ((x_train[:2000].astype("float32") / 127.5) - 1)[..., None]

T = 200
betas = np.linspace(1e-4, 0.02, T).astype("float32")
alphas = 1 - betas
alpha_bar = np.cumprod(alphas).astype("float32")

def build_model():
    inp, t_inp = layers.Input((28, 28, 1)), layers.Input((1,))
    t_emb = layers.Reshape((28, 28, 1))(layers.Dense(28 * 28)(t_inp))
    x = layers.Concatenate()([inp, t_emb])
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    return tf.keras.Model([inp, t_inp], layers.Conv2D(1, 3, padding="same")(x))

model = build_model()
model.compile(optimizer="adam", loss="mse")

# Quick training loop (stands in for loading a pre-trained checkpoint)
for _ in range(3):
    t = np.random.randint(0, T, size=len(x_train))
    noise = np.random.normal(size=x_train.shape).astype("float32")
    ab = alpha_bar[t].reshape(-1, 1, 1, 1)
    x_noisy = np.sqrt(ab) * x_train + np.sqrt(1 - ab) * noise
    model.fit([x_noisy, t.reshape(-1, 1)], noise, batch_size=64, epochs=1, verbose=1)

# Inference: reverse diffusion sampling to generate new images
x = tf.random.normal((16, 28, 28, 1))
for t in reversed(range(T)):
    pred_noise = model([x, tf.fill((16, 1), t)], training=False)
    a, ab = alphas[t], alpha_bar[t]
    x = (1 / np.sqrt(a)) * (x - ((1 - a) / np.sqrt(1 - ab)) * pred_noise)
    if t > 0:
        x += np.sqrt(betas[t]) * tf.random.normal(x.shape)

imgs = (x.numpy().squeeze() + 1) / 2
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i], cmap="gray"); ax.axis("off")
plt.suptitle("Generated MNIST images (DDPM sampling)")
plt.show()
